# Day 43 — Model interpretation: SHAP, PDP, LIME (basics)
Objectives:
- Global vs local interpretability.
- SHAP values for tree/linear models.
- Partial Dependence Plots (PDP).
- LIME overview (optional).
Note: SHAP can be heavy; use small datasets.

In [ ]:
import shap
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr,ytr)
rf.score(Xte,yte)
explainer = shap.TreeExplainer(rf)
X_sample = Xte[:200]
shap_values = explainer.shap_values(X_sample)
# SHAP <0.45 returned a list per class; newer SHAP returns a 3-D array.
if isinstance(shap_values, list):
    positive_class_values = shap_values[1]
elif shap_values.ndim == 3:
    positive_class_values = shap_values[:, :, 1]
else:
    positive_class_values = shap_values
shap.summary_plot(positive_class_values, X_sample, show=False)


## Partial Dependence
Understand marginal effect of features.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
PartialDependenceDisplay.from_estimator(rf, Xtr, [0, (0,1)], ax=ax)
plt.show()


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — global versus local explanations, perturbation assumptions, and causal limits

### Mental model

An explanation describes a fitted model under a chosen data reference;
it does not automatically describe the data-generating process.
**Global** tools summarize behavior across many observations. **Local**
tools allocate one prediction relative to a baseline or nearby cases.

Permutation importance breaks a feature's observed association and
measures score loss. Partial dependence averages predictions while
varying features, potentially creating unrealistic combinations when
features are correlated. SHAP values also require assumptions about
missing-feature dependence and a background distribution.

### Read the API before running it

- **`permutation_importance(..., X_valid, y_valid)`:** measures predictive reliance on held-out data with repeated shuffles and a declared scorer.
- **`PartialDependenceDisplay.from_estimator(...)`:** averages model predictions over a grid while holding the empirical distribution of other features.
- **`shap.TreeExplainer(model, data=background, ...)`:** allocates model output under an explicit dependence/background convention; output shape is model/version dependent.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — measure one local prediction's sensitivity

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The changed feature value combined with all unchanged values represents a plausible input.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, stratify=y, random_state=4301
)
model = RandomForestClassifier(n_estimators=80, random_state=4301, n_jobs=1)
model.fit(X_train, y_train)
row = X_valid[[0]].copy()
baseline = model.predict_proba(row)[0, 1]
changed = row.copy()
changed[0, 0] = np.median(X_train[:, 0])
perturbed = model.predict_proba(changed)[0, 1]
print({"baseline": baseline, "one_feature_changed": perturbed,
       "difference": perturbed - baseline})

**Expected observation:** Changing one feature can move the prediction, but the difference is a model sensitivity under an artificial intervention.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — give permutation importance an uncertainty interval

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Held-out ROC AUC is a valid model-quality measure and the model performs well enough to explain.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_valid, y_valid, scoring="roc_auc",
    n_repeats=8, random_state=4302, n_jobs=1
)
order = result.importances_mean.argsort()[::-1][:3]
summary = [
    (int(i), float(result.importances_mean[i]), float(result.importances_std[i]))
    for i in order
]
print(summary)
assert all(mean >= -3 * std for _, mean, std in summary)

**Expected observation:** Importance is a distribution across shuffles, not one exact ranking; close features may be indistinguishable.

### Debugging and practice ramp

**Common mistake:** Writing 'feature X causes outcome Y' beneath a SHAP, PDP, or importance plot.

**Diagnostic:** State the explained output, dataset/split, background, scorer, perturbation rule, correlation structure, and whether the input combinations are plausible.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define global versus local explanations, perturbation assumptions, and causal limits in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not explain a poorly validated model or expose sensitive row-level explanations without an access/privacy policy.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## LIME (optional)
Install with `pip install lime` and see docs: https://github.com/marcotcr/lime

## Learner exercises and progressive hints

1. Compare a SHAP summary for the top five important features.

**Verify:** For task `Compare a SHAP summary for the top five important features`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






2. Plot PDP for the most important feature and interpret it.

**Verify:** For task `Plot PDP for the most important feature and interpret it`, show the labeled figure and reconcile it with a numeric summary so appearance is not the only check.






3. Optionally use LIME on one prediction and compare it with SHAP.

LIME is installed by the `ml` dependency group but remains an optional lesson
extension. Complete the required work with SHAP and scikit-learn before adding a
second explanation library.

**Verify:** For task `Optionally use LIME on one prediction and compare it with SHAP`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.







### Progressive hints

1. Rank features with mean absolute SHAP values, preserve the original feature
   names, and then limit the display.
2. Choose importance from held-out permutation or aggregated SHAP, not from a
   single local case. Look for regions with little data support.
3. If you intentionally install LIME during a connected session, fix its random
   seed and compare direction, magnitude, and stability—not just wording.

### Additional mastery practice

Treat explanations as model diagnostics tied to a dataset and baseline. Distinguish global from local behavior and predictive association from causation.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Local-versus-global diagnosis:** Construct a case where a feature is globally important but contributes little to one prediction. Explain why those statements do not conflict.
   **Progressive hint:** Global importance aggregates across rows; a local explanation is conditioned on one row and its baseline.

**Verify:** For task `Local-versus-global diagnosis: Construct a case where a feature is globally important but con...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.







5. **Explanation leakage:** Explain why selecting the 'most important' features with the final test set and then retraining a smaller model contaminates evaluation.
   **Progressive hint:** The explanation becomes a supervised feature-selection step. Keep the test set unavailable until the complete selection procedure is frozen.

**Verify:** For task `Explanation leakage: Explain why selecting the 'most important' features with the final test...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.







6. **Correlated-feature and causality check:** Duplicate or strongly correlate one predictor, compare SHAP, PDP, and permutation results, and write a cautious stakeholder explanation.
   **Progressive hint:** Credit can move or split between substitutes; marginal perturbations can create implausible combinations.

**Verify:** For task `Correlated-feature and causality check: Duplicate or strongly correlate one predictor, compar...`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Local-versus-global diagnosis


# Practice 5 — Explanation leakage


# Practice 6 — Correlated-feature and causality check
